In [1]:
from tqdm import tqdm

In [2]:
import requests
url = 'https://rest.uniprot.org/uniprotkb/stream?compressed=false&format=fasta&query=%28%28proteome%3AUP000005640%29%29'
all_fastas = requests.get(url).text

len(all_fastas)

from Bio import SeqIO 
import pandas as pd 
import numpy as np
from io import StringIO

records = SeqIO.parse(StringIO(all_fastas), "fasta") 
ids = [] 
descriptions = [] 
seqs = [] 
for record in records: 
    ids.append(record.id) 
    descriptions.append(record.description) 
    seqs.append(record.seq)

human_uniprot = pd.DataFrame({'id': ids,'description': descriptions, 'seq': seqs})

human_uniprot["seq"] = human_uniprot["seq"].apply(lambda s: s.__str__())

human_uniprot["uniprot_acc"] = human_uniprot["id"].apply(lambda s: s.split("|")[1])

In [3]:
(human_uniprot["uniprot_acc"] == "P61978").sum()

np.int64(1)

In [8]:
human_uniprot.shape

(83607, 4)

In [5]:
## Subset the the human proteome that does not contain phosphosite
import os

In [9]:
rootdir = "${SPARK_ACQUISITION_ROOT}/kinase_ml/data/protein_structures/af2_pdb/"
af2_ls = os.listdir(rootdir)
af2_uniprot_accs = [pdb.split("-")[1] for pdb in af2_ls if (len(pdb.split("-"))>1) and (pdb.endswith('pdb.gz'))]
print(len(af2_uniprot_accs))
af2_paths = [pdb for pdb in af2_ls if (len(pdb.split("-"))>1) and (pdb.endswith('pdb.gz'))]

In [10]:
human_uniprot = human_uniprot.loc[human_uniprot["uniprot_acc"].isin(af2_uniprot_accs),:]

In [13]:
human_uniprot_background = human_uniprot

In [12]:
# human_uniprot_background = human_uniprot.loc[~human_uniprot["uniprot_acc"].isin(psite_all),:]

In [51]:
## Sample S/T/Y surrounding environment as negative samples
def get_background_AA(seq,AA):
    seq_orig = seq[:]
    search_start = 15
    res_s = []
    while (len(seq)>2*search_start + 1):
        try:
            first_res_ind = seq[search_start:].index(AA)
            psite_ind = search_start + first_res_ind
            frag = seq[psite_ind-15:psite_ind+16]
            seq = seq[psite_ind+16:]
#             print(frag)
            psite_ind_orig = seq_orig.index(frag)+15
#             print(psite_ind_orig)
#             print(seq_orig[psite_ind_orig-15:psite_ind_orig+16])
            res_s.append(psite_ind_orig+1)
        except:
            break
    return res_s

In [52]:
# human_uniprot_background.head()

In [53]:
res_s_seq = []
res_s_psites = []
res_s_acc = []
for i in tqdm(range(human_uniprot_background.shape[0])):
    seq = human_uniprot_background["seq"].iloc[i]
    uniprot_id = human_uniprot_background["uniprot_acc"].iloc[i]
    res_s_psite = get_background_AA(seq,'S')
    res_s_seq.append(seq)
    res_s_psites.append(res_s_psite)
    res_s_acc.append(uniprot_id)

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 20426/20426 [00:00<00:00, 25214.69it/s]


In [54]:
res_t_seq = []
res_t_psites = []
res_t_acc = []
for i in tqdm(range(human_uniprot_background.shape[0])):
    seq = human_uniprot_background["seq"].iloc[i]
    uniprot_id = human_uniprot_background["uniprot_acc"].iloc[i]
    res_t_psite = get_background_AA(seq,'T')
    res_t_seq.append(seq)
    res_t_psites.append(res_t_psite)
    res_t_acc.append(uniprot_id)

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 20426/20426 [00:00<00:00, 31448.65it/s]


In [55]:
res_y_seq = []
res_y_psites = []
res_y_acc = []
for i in tqdm(range(human_uniprot_background.shape[0])):
    seq = human_uniprot_background["seq"].iloc[i]
    uniprot_id = human_uniprot_background["uniprot_acc"].iloc[i]
    res_y_psite = get_background_AA(seq,'Y')
    res_y_seq.append(seq)
    res_y_psites.append(res_y_psite)
    res_y_acc.append(uniprot_id)

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 20426/20426 [00:00<00:00, 36493.48it/s]


In [56]:
len(res_y_acc)

20426

In [64]:
#index start from 1

In [57]:
# convert to dataframe
y_background_df = pd.DataFrame({"uniprot":res_y_acc,"seq":res_y_seq,"psites":res_y_psites})
y_background_df = y_background_df.loc[y_background_df["psites"].apply(lambda s: len(s)>0),:]
y_background_df = y_background_df.explode("psites")
y_background_df.columns = ["substrate","substrate_seq","p_ind"]

In [58]:
s_background_df = pd.DataFrame({"uniprot":res_s_acc,"seq":res_s_seq,"psites":res_s_psites})
s_background_df = s_background_df.loc[s_background_df["psites"].apply(lambda s: len(s)>0),:]
s_background_df = s_background_df.explode("psites")
s_background_df.columns = ["substrate","substrate_seq","p_ind"]

In [59]:
t_background_df = pd.DataFrame({"uniprot":res_t_acc,"seq":res_t_seq,"psites":res_t_psites})
t_background_df = t_background_df.loc[t_background_df["psites"].apply(lambda s: len(s)>0),:]
t_background_df = t_background_df.explode("psites")
t_background_df.columns = ["substrate","substrate_seq","p_ind"]

### exclude the known psites from the background

In [27]:
dataroot = "${SPARK_ACQUISITION_ROOT}/kinase_ml/data"
psite_animal = pd.read_pickle(os.path.join(dataroot,"pretraining","phosphosite_animal.pkl"))
psite_xl = pd.read_pickle(os.path.join(dataroot,"pretraining","phosphosite_xl.pkl"))
interaction_exp = pd.read_pickle(os.path.join(dataroot,"interactions","interaction_exp.pkl")) 
interaction_pred = pd.read_pickle(os.path.join(dataroot,"interactions","interaction_pred.pkl"))

In [78]:
interaction_pred

,kinase_acc,substrate_acc,position,residue
0,P04049,P28482,113,Y
1,P04049,P28482,142,S
2,P04049,P28482,181,T
3,P04049,P28482,185,T
4,P04049,P28482,187,Y
...,...,...,...,...
821099,P07947,P11142,638,S
821100,P07947,P11142,64,T
821101,P07947,P11142,641,T
821102,P07947,P11142,66,T


In [81]:
# interaction_pred.apply(lambda s: "-".join([str(s["substrate_acc"]),str(s["position"]),str(s["residue"])]), axis = 1)

In [89]:
exp_psite_ids = interaction_exp.apply(lambda s: "-".join([str(s["substrate"]),str(s["p_ind"])]), axis = 1)

In [90]:
psite_animal_ids = psite_animal.apply(lambda s: "-".join([s["uniprot"],str(s["position"])]), axis = 1)

In [91]:
all_psite_ids = exp_psite_ids.tolist() + psite_animal_ids.tolist()

In [92]:
y_background_df["psite_id"] = y_background_df.apply(lambda s: "-".join([s["substrate"],str(s["p_ind"])]), axis = 1)
s_background_df["psite_id"] = s_background_df.apply(lambda s: "-".join([s["substrate"],str(s["p_ind"])]), axis = 1)
t_background_df["psite_id"] = t_background_df.apply(lambda s: "-".join([s["substrate"],str(s["p_ind"])]), axis = 1)

In [95]:
y_background_df = y_background_df.loc[~y_background_df["psite_id"].isin(all_psite_ids),:]
s_background_df = s_background_df.loc[~s_background_df["psite_id"].isin(all_psite_ids),:]
t_background_df = t_background_df.loc[~t_background_df["psite_id"].isin(all_psite_ids),:]

In [100]:
y_background_df = y_background_df.drop("psite_id",axis = 1)
s_background_df = s_background_df.drop("psite_id",axis = 1)
t_background_df = t_background_df.drop("psite_id",axis = 1)

In [101]:
from Bio import SeqIO 
import pandas as pd 
import numpy as np
from io import StringIO
import pickle
import os
import random
dataroot = "${SPARK_ACQUISITION_ROOT}/kinase_ml/data"

interations = pd.read_pickle("${SPARK_ACQUISITION_ROOT}/kinase_ml/data/interactions/interaction_exp_cleaned_220923.pkl")

interations = interations.reset_index().iloc[:,1:]

interations["substrate_seq"].iloc[0][182]

interations.groupby("p_aa").count()

interations = interations.loc[(interations["p_aa"] == 'S')|(interations["p_aa"] == 'T')|(interations["p_aa"] == 'Y'),:].reset_index()

interations.groupby("p_aa").count()

,index,substrate,kinase,p_ind,substrate_seq,kinase_seq,kinase_pfam,kinase_interpro,domain_cc,domain_ft,protein_families,kinase_flag,kinase_coord,valid_check
p_aa,,,,,,,,,,,,,,
S,18625,18625,18625,18625,18625,18625,18618,18625,6629,18625,18194,18625,18625,18625
T,6553,6553,6553,6553,6553,6553,6531,6553,2697,6553,6397,6553,6553,6553
Y,4716,4716,4716,4716,4716,4716,4716,4716,2346,4716,4611,4716,4716,4716


In [102]:
interations__e2e = interations.loc[:,["substrate","substrate_seq","kinase","kinase_seq","p_ind","p_aa","kinase_coord"]]
interations__e2e['label'] = 1

In [103]:
s_interations = interations__e2e.loc[interations__e2e["p_aa"] == 'S',:]
s_neg_interations = s_background_df.sample(n=s_interations.shape[0],replace = False)
s_neg_interations['label'] = 0
s_neg_interations = pd.concat([s_interations.loc[:,["kinase","kinase_seq","p_aa","kinase_coord"]].reset_index().iloc[:,1:],
           s_neg_interations.reset_index().iloc[:,1:]],
          axis = 1,
         ignore_index = True)
s_neg_interations.columns = ["kinase","kinase_seq","p_aa","kinase_coord","substrate","substrate_seq","p_ind","label"]
s_neg_interations = s_neg_interations.loc[:,["substrate","substrate_seq","kinase","kinase_seq","p_ind","p_aa","kinase_coord","label"]]
s_interations_e2e_1n1p = pd.concat([s_interations,s_neg_interations],axis = 0).reset_index().iloc[:,1:]

In [104]:
t_interations = interations__e2e.loc[interations__e2e["p_aa"] == 'T',:]
t_neg_interations = t_background_df.sample(n=t_interations.shape[0],replace = False)
t_neg_interations['label'] = 0
t_neg_interations = pd.concat([t_interations.loc[:,["kinase","kinase_seq","p_aa","kinase_coord"]].reset_index().iloc[:,1:],
           t_neg_interations.reset_index().iloc[:,1:]],
          axis = 1,
         ignore_index = True)
t_neg_interations.columns = ["kinase","kinase_seq","p_aa","kinase_coord","substrate","substrate_seq","p_ind","label"]
t_neg_interations = t_neg_interations.loc[:,["substrate","substrate_seq","kinase","kinase_seq","p_ind","p_aa","kinase_coord","label"]]
t_interations_e2e_1n1p = pd.concat([t_interations,t_neg_interations],axis = 0).reset_index().iloc[:,1:]

In [105]:
y_interations = interations__e2e.loc[interations__e2e["p_aa"] == 'Y',:]
y_neg_interations = y_background_df.sample(n=y_interations.shape[0],replace = False)
y_neg_interations['label'] = 0
y_neg_interations = pd.concat([y_interations.loc[:,["kinase","kinase_seq","p_aa","kinase_coord"]].reset_index().iloc[:,1:],
           y_neg_interations.reset_index().iloc[:,1:]],
          axis = 1,
         ignore_index = True)
y_neg_interations.columns = ["kinase","kinase_seq","p_aa","kinase_coord","substrate","substrate_seq","p_ind","label"]
y_neg_interations = y_neg_interations.loc[:,["substrate","substrate_seq","kinase","kinase_seq","p_ind","p_aa","kinase_coord","label"]]
y_interations_e2e_1n1p = pd.concat([y_interations,y_neg_interations],axis = 0).reset_index().iloc[:,1:]

In [106]:
dat = pd.concat([s_interations_e2e_1n1p,t_interations_e2e_1n1p,y_interations_e2e_1n1p],axis = 0)

In [107]:
dat.groupby(["label","p_aa"]).count()

substrate  substrate_seq  kinase  kinase_seq  p_ind  kinase_coord
label p_aa                                                                   
0     S         18625          18625   18625       18625  18625         18625
      T          6553           6553    6553        6553   6553          6553
      Y          4716           4716    4716        4716   4716          4716
1     S         18625          18625   18625       18625  18625         18625
      T          6553           6553    6553        6553   6553          6553
      Y          4716           4716    4716        4716   4716          4716

In [108]:
dat.to_pickle("${SPARK_ACQUISITION_ROOT}/kinase_ml/data/interactions/interations_e2e_1n1p_withacc_230616.pkl")

## Class mix-match

In [109]:
interaction_classed = pd.read_pickle("${SPARK_ACQUISITION_ROOT}/kinase_ml/data/interactions/interaction_classed_221017.pkl")

In [110]:
# interaction_classed
unique_psites = interaction_classed.loc[:,["substrate","p_ind"]].drop_duplicates()
def count_family(s):
#     print(s["substrate"])
    return interaction_classed.loc[(interaction_classed["substrate"]==s["substrate"])&(interaction_classed["p_ind"]==s["p_ind"]),["pk_family"]].drop_duplicates().shape[0]

unique_psites["num_fam"] = unique_psites.apply(lambda s: count_family(s), axis = 1)

unique_psites.loc[(unique_psites["num_fam"]>3),:].shape[0]

55

In [111]:
## Negative Pairs Generation 
res = []
num_neg = 1

for i in interaction_classed.index:
    kin_acc = interaction_classed.loc[i,"kinase"]
    kin_seq = interaction_classed.loc[i,"kinase_seq"]
    kin_coord = interaction_classed.loc[i,"kinase_coord"]
    p_aa = interaction_classed.loc[i,"p_aa"] 
    s = interaction_classed.loc[i,:]
    pk_fam = interaction_classed.loc[(interaction_classed["substrate"]==s["substrate"])&(interaction_classed["p_ind"]==s["p_ind"]),"pk_family"].drop_duplicates()
    subfam = interaction_classed.loc[(interaction_classed["substrate"]==s["substrate"])&(interaction_classed["p_ind"]==s["p_ind"]),"pk_subfamily"].drop_duplicates()
    if (p_aa == 'S'):
        bg_seqs = interaction_classed.loc[(~interaction_classed["pk_family"].isin(pk_fam))& \
                                          (interaction_classed["pk_family"]!="Tyr")&
                                          (interaction_classed["p_aa"]==p_aa), \
                        :].sample(num_neg)
        for index,bg_seq in bg_seqs.iterrows():
            res.append({"substrate":bg_seq["substrate"], \
                        "substrate_seq": bg_seq["substrate_seq"], \
                        "kinase":kin_acc, \
                        "kinase_seq":kin_seq, \
                        "p_ind": bg_seq["p_ind"], \
                        "p_aa":bg_seq["p_aa"], \
                        "kinase_coord": kin_coord})
    elif (p_aa == 'T'):
        bg_seqs = interaction_classed.loc[(~interaction_classed["pk_family"].isin(pk_fam))& \
                                          (interaction_classed["pk_family"]!="Tyr")&
                                          (interaction_classed["p_aa"]==p_aa), \
                        :].sample(num_neg)
        for index,bg_seq in bg_seqs.iterrows():
            res.append({"substrate":bg_seq["substrate"], \
                        "substrate_seq": bg_seq["substrate_seq"], \
                        "kinase":kin_acc, \
                        "kinase_seq":kin_seq, \
                        "p_ind": bg_seq["p_ind"], \
                        "p_aa":bg_seq["p_aa"], \
                        "kinase_coord": kin_coord})
    elif (p_aa == 'Y'):
        bg_seqs = interaction_classed.loc[(interaction_classed["pk_family"]=="Tyr")& \
                                          (~interaction_classed["pk_subfamily"].isin(subfam))&
                                          (interaction_classed["p_aa"]==p_aa), \
                        :].sample(num_neg)
        for index,bg_seq in bg_seqs.iterrows():
            res.append({"substrate":bg_seq["substrate"], \
                        "substrate_seq": bg_seq["substrate_seq"], \
                        "kinase":kin_acc, \
                        "kinase_seq":kin_seq, \
                        "p_ind": bg_seq["p_ind"], \
                        "p_aa":bg_seq["p_aa"], \
                        "kinase_coord": kin_coord})

In [112]:
interations__e2e_classed = interaction_classed.loc[:,["substrate","substrate_seq","kinase","kinase_seq","p_ind","p_aa","kinase_coord"]]
interations__e2e_classed['label'] = 1

neg_interations_classed = pd.DataFrame(res)
neg_interations_classed['label'] = 0
interations_e2e_np_classed = pd.concat([interations__e2e_classed,neg_interations_classed],axis = 0).reset_index().iloc[:,1:]

interations_e2e_np_classed.groupby(["p_aa","label"]).count()

substrate  substrate_seq  kinase  kinase_seq  p_ind  kinase_coord
p_aa label                                                                   
S    0          18601          18601   18601       18601  18601         18601
     1          18601          18601   18601       18601  18601         18601
T    0           6550           6550    6550        6550   6550          6550
     1           6550           6550    6550        6550   6550          6550
Y    0           4689           4689    4689        4689   4689          4689
     1           4689           4689    4689        4689   4689          4689

In [118]:
interations_e2e_np_classed.head()

,substrate,substrate_seq,kinase,kinase_seq,p_ind,p_aa,kinase_coord,label
0,P34901,MAPVCLFAPLLLLLLGGFPVAPGESIRETEVIDPQDLLEGRYFSGA...,Q05655,MAPFLRIAFNSYELGSLQAEDEANQPFCAVKMKEALSTERGKTLVQ...,183,S,"[[349, 603]]",1
1,Q9UQL6,MNSPNESDGMSGREPSLEILPRTSLHSIPVTVEVKPVLPRAMPSSM...,Q05655,MAPFLRIAFNSYELGSLQAEDEANQPFCAVKMKEALSTERGKTLVQ...,259,S,"[[349, 603]]",1
2,P18433-2,MDSWFILVLLGSGLICVSANNATTVAPSVGITRLINSSTAEPVKEE...,Q05655,MAPFLRIAFNSYELGSLQAEDEANQPFCAVKMKEALSTERGKTLVQ...,204,S,"[[349, 603]]",1
3,P18433-2,MDSWFILVLLGSGLICVSANNATTVAPSVGITRLINSSTAEPVKEE...,Q05655,MAPFLRIAFNSYELGSLQAEDEANQPFCAVKMKEALSTERGKTLVQ...,180,S,"[[349, 603]]",1
4,P61978,METEQPEETFPNTETNGEFGKRPAEDMEEEQAFKRSRNTDEMVELR...,Q05655,MAPFLRIAFNSYELGSLQAEDEANQPFCAVKMKEALSTERGKTLVQ...,302,S,"[[349, 603]]",1


In [119]:
interations_e2e_np = pd.read_pickle("${SPARK_ACQUISITION_ROOT}/kinase_ml/data/interactions/interations_e2e_1n1p_withacc_230616.pkl")

In [120]:
interations_e2e_np.head()

,substrate,substrate_seq,kinase,kinase_seq,p_ind,p_aa,kinase_coord,label
0,P34901,MAPVCLFAPLLLLLLGGFPVAPGESIRETEVIDPQDLLEGRYFSGA...,Q05655,MAPFLRIAFNSYELGSLQAEDEANQPFCAVKMKEALSTERGKTLVQ...,183,S,"[[349, 603]]",1
1,Q9UQL6,MNSPNESDGMSGREPSLEILPRTSLHSIPVTVEVKPVLPRAMPSSM...,Q05655,MAPFLRIAFNSYELGSLQAEDEANQPFCAVKMKEALSTERGKTLVQ...,259,S,"[[349, 603]]",1
2,P18433-2,MDSWFILVLLGSGLICVSANNATTVAPSVGITRLINSSTAEPVKEE...,Q05655,MAPFLRIAFNSYELGSLQAEDEANQPFCAVKMKEALSTERGKTLVQ...,204,S,"[[349, 603]]",1
3,P18433-2,MDSWFILVLLGSGLICVSANNATTVAPSVGITRLINSSTAEPVKEE...,Q05655,MAPFLRIAFNSYELGSLQAEDEANQPFCAVKMKEALSTERGKTLVQ...,180,S,"[[349, 603]]",1
4,P61978,METEQPEETFPNTETNGEFGKRPAEDMEEEQAFKRSRNTDEMVELR...,Q05655,MAPFLRIAFNSYELGSLQAEDEANQPFCAVKMKEALSTERGKTLVQ...,302,S,"[[349, 603]]",1


In [116]:
# TODO: Edit to the proper way
interations_e2e_np_classed.to_pickle("${SPARK_ACQUISITION_ROOT}/kinase_ml/data/interactions/interations_e2e_1n1p_classed_withacc_230621.pkl")
interations_e2e_np.to_pickle("${SPARK_ACQUISITION_ROOT}/kinase_ml/data/interactions/interations_e2e_1n1p_random_withacc_230621.pkl")
interations_e2e_np_classed_random = pd.concat([interations_e2e_np_classed,interations_e2e_np],axis = 0).reset_index().iloc[:,1:]
interations_e2e_np_classed_random.to_pickle("${SPARK_ACQUISITION_ROOT}/kinase_ml/data/interactions/interations_e2e_1n1p_classed_random_withacc_230621.pkl")

In [2]:
# interations_e2e_np_classed_random.groupby(['p_aa','label']).count()

In [121]:
! cp ${SPARK_ACQUISITION_ROOT}/kinase_ml/data/interactions/interations_e2e_1n1p_classed_random_withacc_230621.pkl ../../data/training_data/

In [ ]:
interations_e2e_np_classed_random.head()

## Collect 1N1P for LysateKSI

In [4]:
import pandas as pd
interations_e2e_np_classed_random = pd.read_pickle("${SPARK_ACQUISITION_ROOT}/kinase_ml/data/interactions/interations_e2e_1n1p_classed_random_withacc_230621.pkl")

In [5]:
dataroot = "${SPARK_ACQUISITION_ROOT}/kinase_ml/data"

interations = pd.read_pickle("${SPARK_ACQUISITION_ROOT}/kinase_ml/data/interactions/lysate_ksi_cleaned_221115.pkl")

interations = interations.reset_index().iloc[:,1:]
interations.groupby("p_aa").count()

,pk_family,kinase_name,substrate,substrate_name,psite,SIDIC,p_threshold,substrate_seq,kinase_mutation,kinase,kinase_seq,Domain [FT],kinase_coord,p_ind
p_aa,,,,,,,,,,,,,,
S,41382,41382,41382,41382,41382,41382,41382,41382,41382,41382,41382,41382,41382,41382
T,18395,18395,18395,18395,18395,18395,18395,18395,18395,18395,18395,18395,18395,18395
Y,84059,84059,84059,84059,84059,84059,84059,84059,84059,84059,84059,84059,84059,84059


In [123]:
interations__e2e = interations.loc[:,["substrate","substrate_seq","kinase","kinase_seq","p_ind","p_aa","kinase_coord"]]
interations__e2e['label'] = 1

In [124]:
s_interations = interations__e2e.loc[interations__e2e["p_aa"] == 'S',:]
s_neg_interations = s_background_df.sample(n=s_interations.shape[0],replace = False)
s_neg_interations['label'] = 0
s_neg_interations = pd.concat([s_interations.loc[:,["kinase","kinase_seq","p_aa","kinase_coord"]].reset_index().iloc[:,1:],
           s_neg_interations.reset_index().iloc[:,1:]],
          axis = 1,
         ignore_index = True)
s_neg_interations.columns = ["kinase","kinase_seq","p_aa","kinase_coord","substrate","substrate_seq","p_ind","label"]
s_neg_interations = s_neg_interations.loc[:,["substrate","substrate_seq","kinase","kinase_seq","p_ind","p_aa","kinase_coord","label"]]
s_interations_e2e_1n1p = pd.concat([s_interations,s_neg_interations],axis = 0).reset_index().iloc[:,1:]

t_interations = interations__e2e.loc[interations__e2e["p_aa"] == 'T',:]
t_neg_interations = t_background_df.sample(n=t_interations.shape[0],replace = False)
t_neg_interations['label'] = 0
t_neg_interations = pd.concat([t_interations.loc[:,["kinase","kinase_seq","p_aa","kinase_coord"]].reset_index().iloc[:,1:],
           t_neg_interations.reset_index().iloc[:,1:]],
          axis = 1,
         ignore_index = True)
t_neg_interations.columns = ["kinase","kinase_seq","p_aa","kinase_coord","substrate","substrate_seq","p_ind","label"]
t_neg_interations = t_neg_interations.loc[:,["substrate","substrate_seq","kinase","kinase_seq","p_ind","p_aa","kinase_coord","label"]]
t_interations_e2e_1n1p = pd.concat([t_interations,t_neg_interations],axis = 0).reset_index().iloc[:,1:]

y_interations = interations__e2e.loc[interations__e2e["p_aa"] == 'Y',:]
y_neg_interations = y_background_df.sample(n=y_interations.shape[0],replace = False)
y_neg_interations['label'] = 0
y_neg_interations = pd.concat([y_interations.loc[:,["kinase","kinase_seq","p_aa","kinase_coord"]].reset_index().iloc[:,1:],
           y_neg_interations.reset_index().iloc[:,1:]],
          axis = 1,
         ignore_index = True)
y_neg_interations.columns = ["kinase","kinase_seq","p_aa","kinase_coord","substrate","substrate_seq","p_ind","label"]
y_neg_interations = y_neg_interations.loc[:,["substrate","substrate_seq","kinase","kinase_seq","p_ind","p_aa","kinase_coord","label"]]
y_interations_e2e_1n1p = pd.concat([y_interations,y_neg_interations],axis = 0).reset_index().iloc[:,1:]

In [125]:
dat = pd.concat([s_interations_e2e_1n1p,t_interations_e2e_1n1p,y_interations_e2e_1n1p],axis = 0)
dat.groupby(["label","p_aa"]).count()

substrate  substrate_seq  kinase  kinase_seq  p_ind  kinase_coord
label p_aa                                                                   
0     S         41382          41382   41382       41382  41382         41382
      T         18395          18395   18395       18395  18395         18395
      Y         84059          84059   84059       84059  84059         84059
1     S         41382          41382   41382       41382  41382         41382
      T         18395          18395   18395       18395  18395         18395
      Y         84059          84059   84059       84059  84059         84059

In [126]:
dat.head()

,substrate,substrate_seq,kinase,kinase_seq,p_ind,p_aa,kinase_coord,label
0,1433B_HUMAN,MTMDKSELVQKAKLAEQAERYDDMAAAMKAVTEQGHELSNEERNLL...,P00519,MLEICLKLVGCKSKKGLSSSSSCYLEEALQRPVASDFEPQGLSEAA...,212,S,"[[242, 493]]",1
1,1433G_HUMAN,MVDREQLVQKARLAEQAERYDDMAAAMKNVTELNEPLSNEERNLLS...,P00519,MLEICLKLVGCKSKKGLSSSSSCYLEEALQRPVASDFEPQGLSEAA...,215,S,"[[242, 493]]",1
2,ABL1_HUMAN,MLEICLKLVGCKSKKGLSSSSSCYLEEALQRPVASDFEPQGLSEAA...,P00519,MLEICLKLVGCKSKKGLSSSSSCYLEEALQRPVASDFEPQGLSEAA...,1106,S,"[[242, 493]]",1
3,ABL1_HUMAN,MLEICLKLVGCKSKKGLSSSSSCYLEEALQRPVASDFEPQGLSEAA...,P00519,MLEICLKLVGCKSKKGLSSSSSCYLEEALQRPVASDFEPQGLSEAA...,553,S,"[[242, 493]]",1
4,ABL1_HUMAN,MLEICLKLVGCKSKKGLSSSSSCYLEEALQRPVASDFEPQGLSEAA...,P00519,MLEICLKLVGCKSKKGLSSSSSCYLEEALQRPVASDFEPQGLSEAA...,559,S,"[[242, 493]]",1


In [127]:
# dat.to_pickle("${SPARK_ACQUISITION_ROOT}/kinase_ml/data/interactions/interations_e2e_1n1p_db_classed_random_lysate_random_withacc_230621.pkl")

In [6]:
# dat = pd.read_pickle("${SPARK_ACQUISITION_ROOT}/kinase_ml/data/interactions/interations_e2e_1n1p_db_classed_random_lysate_random_withacc_230621.pkl")

In [7]:
dat.to_pickle("${SPARK_ACQUISITION_ROOT}/kinase_ml/data/interactions/interations_e2e_1n1p_lysate_random_withacc_230621.pkl")

In [12]:
dat_final = pd.concat([interations_e2e_np_classed_random,interations_e2e_np_classed_random,dat],axis = 0)

In [13]:
dat_final.to_pickle("${SPARK_ACQUISITION_ROOT}/kinase_ml/data/interactions/interations_e2e_1n1p_db_classed_random_lysate_random_withacc_230621.pkl")


In [14]:
cp -r ${SPARK_ACQUISITION_ROOT}/kinase_ml/data/interactions/interations_e2e_1n1p_db_classed_random_lysate_random_withacc_230621.pkl ../../data/training_data/